# RAG Assignment: Baseline Retrieval vs Cross-Encoder Re-Ranking
**Document:** FIDE Laws of Chess (25 pages, `data/laws_of_chess.pdf`). It is full of exceptions, penalties and "not allowed" clauses, which makes it a good stress test.

**Pipeline 1:** bi-encoder top-10 -> top-3 -> LLM (Answer 1)
**Pipeline 2:** same top-10 -> cross-encoder re-rank -> top-3 -> LLM (Answer 2)

Both pipelines share the same chunks, the same candidate set and the exact same prompt.

## 1. Setup

In [1]:
# pip install sentence-transformers faiss-cpu pypdf pandas torch google-genai
# The Gemini key is read from the GEMINI_API_KEY environment variable (never hard-code it).
import os
import re, time, textwrap
import numpy as np, pandas as pd, faiss, torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder

PDF_PATH   = "data/laws_of_chess.pdf"
BI_MODEL   = "sentence-transformers/all-MiniLM-L6-v2"          # bi-encoder
CE_MODEL   = "cross-encoder/ms-marco-MiniLM-L-6-v2"            # cross-encoder
LLM_MODEL  = "gemini-3.1-flash-lite-preview"                         # Gemini API
CHUNK_WORDS, STRIDE = 100, 80                                   # 20-word overlap
TOP_K, TOP_N = 10, 3
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)

/Users/rick/code/IIIT/SEM3/lma/AS1/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1019 15:11:54.850000 69160 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## 2. Load PDF and chunk it
Fixed-size word windows with a small overlap, remembering the page number of each chunk. Both pipelines use this identical corpus.

In [2]:
words = []
for pno, page in enumerate(PdfReader(PDF_PATH).pages, 1):
    for w in re.sub(r"\s+", " ", page.extract_text() or "").split(" "):
        if w: words.append((w, pno))

chunks = []   # list of dicts: id, page, text
for i in range(0, len(words), STRIDE):
    seg = words[i:i + CHUNK_WORDS]
    chunks.append({"id": len(chunks), "page": seg[0][1], "text": " ".join(w for w, _ in seg)})
    if i + CHUNK_WORDS >= len(words): break
print(f"{len(words)} words -> {len(chunks)} chunks (about {CHUNK_WORDS} words each)")

9662 words -> 121 chunks (about 100 words each)


## 3. Pipeline 1: dense index (Bi-Encoder + FAISS inner product)
Embeddings are L2-normalised, so inner product equals cosine similarity.

In [3]:
bi_encoder = SentenceTransformer(BI_MODEL)
doc_emb = bi_encoder.encode([c["text"] for c in chunks], normalize_embeddings=True, show_progress_bar=False)
index = faiss.IndexFlatIP(doc_emb.shape[1]); index.add(doc_emb)

def retrieve_baseline(query, k=TOP_K):
    q = bi_encoder.encode([query], normalize_embeddings=True)
    scores, ids = index.search(q, k)
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0])]     # ranked best first

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 20343.44it/s]

## 4. Pipeline 2: Cross-Encoder re-ranking
Takes the *same* 10 candidates, scores every (query, chunk) pair jointly and re-sorts.

In [4]:
cross_encoder = CrossEncoder(CE_MODEL)

def rerank(query, candidates):
    pairs = [(query, chunks[i]["text"]) for i, _ in candidates]
    ce_scores = cross_encoder.predict(pairs, show_progress_bar=False)
    order = np.argsort(-ce_scores)
    return [(candidates[j][0], float(ce_scores[j])) for j in order]    # ranked best first

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6955.51it/s]

## 5. LLM generation (identical prompt for both answers)

In [5]:
from google import genai
from google.genai import types
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

PROMPT = '''You are answering questions about the FIDE Laws of Chess.
Use ONLY the context below. If the context does not contain the answer, say "Not found in the provided context."
Be concise (max 3 sentences).

Context:
{context}

Question: {question}
Answer:'''

import json, hashlib
CACHE_FILE = "cache/llm_cache.json"          # free-tier quota is small, so answers are cached between runs
cache = json.load(open(CACHE_FILE)) if os.path.exists(CACHE_FILE) else {}

def generate(question, chunk_ids, use_cache=True):
    context = "\n\n".join(f"[Chunk {i}, p.{chunks[i]['page']}] {chunks[i]['text']}" for i in chunk_ids)
    prompt = PROMPT.format(context=context, question=question)
    key = hashlib.md5((LLM_MODEL + prompt).encode()).hexdigest()
    if use_cache and key in cache: return cache[key]
    for attempt in range(3):                      # retry transient API errors
        try:
            r = client.models.generate_content(model=LLM_MODEL, contents=prompt,
                    config=types.GenerateContentConfig(temperature=0, max_output_tokens=2000))
            cache[key] = (r.text or "").strip()
            json.dump(cache, open(CACHE_FILE, "w"))
            return cache[key]
        except Exception as e:
            err = e
            if "429" in str(e): break             # quota exhausted, retrying will not help
            time.sleep(2 * (attempt + 1))
    return f"[LLM error: {str(err)[:150]}]"

## 6. Stress-test queries
For each query I mark the *gold* chunk(s) by a phrase that must appear in the clause that really answers it (found by reading the rulebook, not by running the models). The notebook then reports where the gold chunk ranks under the bi-encoder and under the cross-encoder.

In [6]:
TESTS = [
 # (category, query, gold phrase present in the answering clause)
 ("Buried chunk (gold at rank 4)",            "How many extra minutes does the opponent get when the arbiter rejects a draw claim?", "rejected the claim, the opponent shall be awarded two extra minutes"),
 ("Exception / corner case",                  "If my opponent's time runs out, do I always win?", "cannot win the game by any series of legal moves"),
 ("Exception / corner case",                  "Does an illegal move always cost the player?", "the game is drawn if the position is such that the opponent cannot checkmate"),
 ("Negation / constraint",                    "Can I still claim a draw after touching a piece?", "loses the right to claim"),
 ("Negation / constraint",                    "Can the scoresheet be used for personal notes?", "scoresheet shall be used only for recording"),
 ("Keyword trap / synonyms (re-ranker helps)", "What can the referee do to a player who breaks the rules?", "arbiter can apply one or more of the following penalties"),
 ("Keyword trap / synonyms (re-ranker hurts)", "How long can a player be absent before the game is forfeited?", "default time"),
 ("Control (easy query, no shift expected)",  "What happens if a player makes a third illegal move?", "third illegal move by the same player"),
 ("Control (negation, already rank 1)",       "Is a player allowed to withdraw a draw claim?", "not allowed to withdraw his claim"),
]

norm = lambda s: re.sub(r"\s+", " ", s).lower()
def gold_ids(phrase):
    return {c["id"] for c in chunks if norm(phrase) in norm(c["text"])}
for cat, q, g in TESTS:
    assert gold_ids(g), f"gold phrase not found: {g}"
print("all gold phrases located in the document")

all gold phrases located in the document


In [7]:
def best_rank(ranking, gold):
    for r, (cid, _) in enumerate(ranking, 1):
        if cid in gold: return r
    return None     # gold chunk not even in the top 10

results = []
for cat, q, g in TESTS:
    gold = gold_ids(g)
    base = retrieve_baseline(q)
    rer  = rerank(q, base)
    a1 = generate(q, [i for i, _ in base[:TOP_N]])
    a2 = generate(q, [i for i, _ in rer[:TOP_N]])
    br, rr = best_rank(base, gold), best_rank(rer, gold)
    results.append(dict(cat=cat, q=q, gold=sorted(gold), base=base, rer=rer, bi_rank=br, ce_rank=rr, a1=a1, a2=a2))
    print("="*100); print(f"[{cat}] {q}")
    print(f"gold chunk(s) {sorted(gold)}:  bi-encoder rank #{br}  ->  cross-encoder rank #{rr}")
    print("bi top-3 :", [i for i, _ in base[:3]], "| CE top-3:", [i for i, _ in rer[:3]])
    print("\nAnswer 1 (baseline):\n" + textwrap.fill(a1, 110))
    print("\nAnswer 2 (re-ranked):\n" + textwrap.fill(a2, 110))

[Buried chunk (gold at rank 4)] How many extra minutes does the opponent get when the arbiter rejects a draw claim?
gold chunk(s) [63]:  bi-encoder rank #4  ->  cross-encoder rank #1
bi top-3 : [62, 61, 59] | CE top-3: [63, 62, 61]

Answer 1 (baseline):
The opponent shall be awarded two extra minutes.

Answer 2 (re-ranked):
The opponent shall be awarded two extra minutes.
[Exception / corner case] If my opponent's time runs out, do I always win?
gold chunk(s) [65, 66]:  bi-encoder rank #10  ->  cross-encoder rank #1
bi top-3 : [33, 63, 61] | CE top-3: [66, 63, 33]

Answer 1 (baseline):
Not found in the provided context.

Answer 2 (re-ranked):
If the opponent cannot win the game by any series of legal moves, the score shall be a draw.


[Exception / corner case] Does an illegal move always cost the player?
gold chunk(s) [45]:  bi-encoder rank #5  ->  cross-encoder rank #3
bi top-3 : [25, 22, 78] | CE top-3: [22, 75, 45]

Answer 1 (baseline):
Not found in the provided context.

Answer 2 (re-ranked):
No. For the first two illegal moves, the arbiter adds two minutes to the opponent's clock, and the game is
only declared lost upon the third illegal move. Additionally, the game is drawn if the opponent cannot
checkmate the player’s king by any possible series of legal moves.


[Negation / constraint] Can I still claim a draw after touching a piece?
gold chunk(s) [58]:  bi-encoder rank #9  ->  cross-encoder rank #1
bi top-3 : [55, 59, 53] | CE top-3: [58, 59, 61]

Answer 1 (baseline):
Not found in the provided context.

Answer 2 (re-ranked):
If you touch a piece as in Article 4.3 without having claimed the draw, you lose the right to claim a draw on
that move.


[Negation / constraint] Can the scoresheet be used for personal notes?
gold chunk(s) [66]:  bi-encoder rank #6  ->  cross-encoder rank #1
bi top-3 : [48, 49, 51] | CE top-3: [66, 80, 48]

Answer 1 (baseline):
Not found in the provided context.

Answer 2 (re-ranked):
Not found in the provided context.


[Keyword trap / synonyms (re-ranker helps)] What can the referee do to a player who breaks the rules?
gold chunk(s) [69]:  bi-encoder rank #9  ->  cross-encoder rank #2
bi top-3 : [70, 61, 26] | CE top-3: [70, 69, 67]

Answer 1 (baseline):
The arbiter may apply penalties including a warning, adjusting the remaining time for either player, declaring
the game lost, reducing or increasing points, or expulsion from the event.

Answer 2 (re-ranked):
The arbiter may impose penalties including a warning, increasing the opponent's time, reducing the offending
player's time, declaring the game lost, adjusting points, or expulsion from the event. Persistent refusal to
comply with the Laws of Chess shall be penalized by loss of the game.
[Keyword trap / synonyms (re-ranker hurts)] How long can a player be absent before the game is forfeited?
gold chunk(s) [32, 33]:  bi-encoder rank #5  ->  cross-encoder rank #9
bi top-3 : [117, 63, 118] | CE top-3: [117, 63, 118]

Answer 1 (baseline):
A player lo

[Control (negation, already rank 1)] Is a player allowed to withdraw a draw claim?
gold chunk(s) [59]:  bi-encoder rank #1  ->  cross-encoder rank #1
bi top-3 : [59, 55, 53] | CE top-3: [59, 54, 53]

Answer 1 (baseline):
No, a player is not allowed to withdraw their claim.

Answer 2 (re-ranked):
No, a player is not allowed to withdraw a draw claim.


### Rank-shift summary

In [8]:
summary = pd.DataFrame([{ "category": r["cat"], "query": r["q"], "gold chunk(s)": r["gold"],
    "bi-encoder rank": r["bi_rank"], "cross-encoder rank": r["ce_rank"],
    "in top-3 before": (r["bi_rank"] or 99) <= 3, "in top-3 after": (r["ce_rank"] or 99) <= 3} for r in results])
summary

,category,query,gold chunk(s),bi-encoder rank,cross-encoder rank,in top-3 before,in top-3 after
0,Buried chunk (gold at rank 4),How many extra minutes does the opponent get when the arbiter rejects a draw claim?,[63],4,1,False,True
1,Exception / corner case,"If my opponent's time runs out, do I always win?","[65, 66]",10,1,False,True
2,Exception / corner case,Does an illegal move always cost the player?,[45],5,3,False,True
3,Negation / constraint,Can I still claim a draw after touching a piece?,[58],9,1,False,True
4,Negation / constraint,Can the scoresheet be used for personal notes?,[66],6,1,False,True
5,Keyword trap / synonyms (re-ranker helps),What can the referee do to a player who breaks the rules?,[69],9,2,False,True
6,Keyword trap / synonyms (re-ranker hurts),How long can a player be absent before the game is forfeited?,"[32, 33]",5,9,False,False
7,"Control (easy query, no shift expected)",What happens if a player makes a third illegal move?,[45],1,1,True,True
8,"Control (negation, already rank 1)",Is a player allowed to withdraw a draw claim?,[59],1,1,True,True


### Full candidate re-ordering for each query (chunk id: bi rank -> CE rank)

In [9]:
for r in results:
    bi_pos = {cid: k for k, (cid, _) in enumerate(r["base"], 1)}
    row = [f"chunk {cid}{'*' if cid in r['gold'] else ''}: #{bi_pos[cid]}->#{k}" for k, (cid, _) in enumerate(r["rer"], 1)]
    print(r["q"]); print("   " + " | ".join(row)); print("   (* = contains the gold clause)\n")

How many extra minutes does the opponent get when the arbiter rejects a draw claim?
   chunk 63*: #4->#1 | chunk 62: #1->#2 | chunk 61: #2->#3 | chunk 59: #3->#4 | chunk 113: #8->#5 | chunk 66: #9->#6 | chunk 53: #6->#7 | chunk 36: #7->#8 | chunk 120: #10->#9 | chunk 52: #5->#10
   (* = contains the gold clause)

If my opponent's time runs out, do I always win?
   chunk 66*: #10->#1 | chunk 63: #2->#2 | chunk 33: #1->#3 | chunk 61: #3->#4 | chunk 34: #6->#5 | chunk 59: #4->#6 | chunk 37: #8->#7 | chunk 110: #9->#8 | chunk 29: #5->#9 | chunk 32: #7->#10
   (* = contains the gold clause)

Does an illegal move always cost the player?
   chunk 22: #2->#1 | chunk 75: #4->#2 | chunk 45*: #5->#3 | chunk 25: #1->#4 | chunk 78: #3->#5 | chunk 44: #10->#6 | chunk 26: #8->#7 | chunk 60: #9->#8 | chunk 47: #6->#9 | chunk 57: #7->#10
   (* = contains the gold clause)

Can I still claim a draw after touching a piece?
   chunk 58*: #9->#1 | chunk 59: #2->#2 | chunk 61: #4->#3 | chunk 20: #7->#4 | chu

## 7. Latency profiling
Retrieval and re-ranking are averaged over 10 repeats per query. LLM generation is timed with one uncached call, because it is identical in both pipelines and the API quota is limited.

In [10]:
def timeit(fn, reps=10):
    ts = []
    for _ in range(reps):
        t = time.perf_counter(); fn(); ts.append((time.perf_counter() - t) * 1000)
    return np.mean(ts), np.std(ts)

queries = [q for _, q, _ in TESTS]
_ = [retrieve_baseline(q) for q in queries[:2]]; _ = rerank(queries[0], retrieve_baseline(queries[0]))   # warm-up

t_base = [timeit(lambda q=q: retrieve_baseline(q))[0] for q in queries]
cands  = [retrieve_baseline(q) for q in queries]
t_rr   = [timeit(lambda q=q, c=c: rerank(q, c))[0] for q, c in zip(queries, cands)]
t_llm  = [np.nan] * len(queries)
t = time.perf_counter(); _ = generate(queries[0], [i for i, _ in cands[0][:3]], use_cache=False); t_llm[0] = (time.perf_counter() - t) * 1000   # one uncached call (API quota is limited)

lat = pd.DataFrame({"query": [q[:60] for q in queries], "Pipeline 1: bi-encoder retrieval (ms)": t_base,
                    "Re-ranking step: 10 CE pairs (ms)": t_rr, "LLM generation (ms)": t_llm})
lat["Pipeline 2 retrieval total (ms)"] = lat.iloc[:, 1] + lat.iloc[:, 2]
display(lat.round(1))
print("MEAN  baseline retrieval : %.1f ms" % np.mean(t_base))
print("MEAN  re-ranking step    : %.1f ms" % np.mean(t_rr))
print("MEAN  pipeline 2 retrieval (bi + CE): %.1f ms  (%.1fx baseline)" % (np.mean(t_base)+np.mean(t_rr), (np.mean(t_base)+np.mean(t_rr))/np.mean(t_base)))
print("LLM generation (single timed call, identical for both pipelines): %.0f ms" % t_llm[0])

# scaling illustration: cross-encoder cost over the *whole* corpus
all_pairs = [(queries[0], c["text"]) for c in chunks]
t = time.perf_counter(); cross_encoder.predict(all_pairs, show_progress_bar=False); full = (time.perf_counter()-t)*1000
print(f"\nCross-encoder over ALL {len(chunks)} chunks for one query: {full:.0f} ms  (vs {np.mean(t_rr):.0f} ms for 10)")

,query,Pipeline 1: bi-encoder retrieval (ms),Re-ranking step: 10 CE pairs (ms),LLM generation (ms),Pipeline 2 retrieval total (ms)
0,How many extra minutes does the opponent get when the arbite,5.9,30.1,690.0,36.1
1,"If my opponent's time runs out, do I always win?",24.2,26.8,NaN,51.1
2,Does an illegal move always cost the player?,7.8,25.7,NaN,33.5
3,Can I still claim a draw after touching a piece?,9.0,26.5,NaN,35.4
4,Can the scoresheet be used for personal notes?,5.8,34.1,NaN,39.9
5,What can the referee do to a player who breaks the rules?,5.0,28.8,NaN,33.9
6,How long can a player be absent before the game is forfeited,5.1,28.3,NaN,33.4
7,What happens if a player makes a third illegal move?,4.9,26.6,NaN,31.5
8,Is a player allowed to withdraw a draw claim?,4.8,26.8,NaN,31.6


MEAN  baseline retrieval : 8.1 ms
MEAN  re-ranking step    : 28.2 ms
MEAN  pipeline 2 retrieval (bi + CE): 36.3 ms  (4.5x baseline)
LLM generation (single timed call, identical for both pipelines): 690 ms



Cross-encoder over ALL 121 chunks for one query: 381 ms  (vs 28 ms for 10)


## 8. Report notes
The written report (architecture comparison, qualitative analysis, latency table, scaling answer) is in `report/RAG_Report.pdf`; the numbers in it come from the cells above.